A website chatbot that can answer questions about the website's content and provide assistance to users. The chatbot will be integrated into the website's interface and will use natural language processing to understand user queries and provide relevant responses. It will also have the ability to learn from user interactions and improve its responses over time.

In [11]:
%pip install -q -U langchain langchain-community langchain-openai langchain-huggingface langchain-text-splitters langchain-classic transformers pinecone huggingface_hub unstructured



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 101.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 141.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.2/347.2 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-adk 1.26.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incom

In [12]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Pinecone
import pinecone
from langchain_classic.chains import RetrievalQAWithSourcesChain
from transformers import pipeline
from huggingface_hub import notebook_login
import textwrap
import sys
import os
import torch

In [13]:
import nltk
nltk.download('punkt')
nltk.download('averged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Error loading averged_perceptron_tagger: Package
[nltk_data]     'averged_perceptron_tagger' not found in index


False

In [24]:
URLs = [
    "https://docs.python.org/3/tutorial/index.html",
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "https://en.wikipedia.org/wiki/Vector_database",
]

In [25]:
loaders=UnstructuredURLLoader(urls=URLs)
data=loaders.load()

In [26]:
data

[Document(metadata={'source': 'https://docs.python.org/3/tutorial/index.html'}, page_content='Navigation\n\nindex\n\nmodules |\n\nnext |\n\nprevious |\n\nPython logo\n\nPython »\n\n3.14.3 Documentation »\n\nThe Python Tutorial\n\n|\n\nThe Python Tutorial¶\n\nTip\n\nThis tutorial is designed for programmers that are new to the Python language, not beginners who are new to programming.\n\nPython is an easy to learn, powerful programming language. It has efficient high-level data structures and a simple but effective approach to object-oriented programming. Python’s elegant syntax and dynamic typing, together with its interpreted nature, make it an ideal language for scripting and rapid application development in many areas on most platforms.\n\nThe Python interpreter and the extensive standard library are freely available in source or binary form for all major platforms from the Python website, https://www.python.org/, and may be freely distributed. The same site also contains distributi

In [27]:
for i, doc in enumerate(data, start=1):
    source = doc.metadata.get("source", "unknown")
    print(f"\n{'='*80}")
    print(f"Document {i} | Source: {source}")
    print(f"{'='*80}\n")
    print(doc.page_content)


Document 1 | Source: https://docs.python.org/3/tutorial/index.html

Navigation

index

modules |

next |

previous |

Python logo

Python »

3.14.3 Documentation »

The Python Tutorial

|

The Python Tutorial¶

Tip

This tutorial is designed for programmers that are new to the Python language, not beginners who are new to programming.

Python is an easy to learn, powerful programming language. It has efficient high-level data structures and a simple but effective approach to object-oriented programming. Python’s elegant syntax and dynamic typing, together with its interpreted nature, make it an ideal language for scripting and rapid application development in many areas on most platforms.

The Python interpreter and the extensive standard library are freely available in source or binary form for all major platforms from the Python website, https://www.python.org/, and may be freely distributed. The same site also contains distributions of and pointers to many free third party Python m

In [28]:
len(data)

3

In [29]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(data)

In [30]:
texts

[Document(metadata={'source': 'https://docs.python.org/3/tutorial/index.html'}, page_content='Navigation\n\nindex\n\nmodules |\n\nnext |\n\nprevious |\n\nPython logo\n\nPython »\n\n3.14.3 Documentation »\n\nThe Python Tutorial\n\n|\n\nThe Python Tutorial¶\n\nTip\n\nThis tutorial is designed for programmers that are new to the Python language, not beginners who are new to programming.\n\nPython is an easy to learn, powerful programming language. It has efficient high-level data structures and a simple but effective approach to object-oriented programming. Python’s elegant syntax and dynamic typing, together with its interpreted nature, make it an ideal language for scripting and rapid application development in many areas on most platforms.\n\nThe Python interpreter and the extensive standard library are freely available in source or binary form for all major platforms from the Python website, https://www.python.org/, and may be freely distributed. The same site also contains distributi

In [ ]:
embeddings = HuggingFaceEmbeddings()

In [ ]:
query_result=embeddings.embed_query("hello world")
len(query_result)

In [ ]:
query_result